# Chapter 9 — Metrics Deep Dive
**Track:** ML from Scratch · California Housing — high-value classifier (Ch.2) + price regressor (Ch.1)

## Core Idea

Every metric answers a **different question**. Picking the wrong one makes a bad model look good.

```
Accuracy  → what fraction of all predictions are right?  (misleading under imbalance)
Precision → of all positive predictions, how many are correct?  (FP cost)
Recall    → of all actual positives, how many did we catch?     (FN cost)
F1        → harmonic mean of P and R  (balanced single score)
AUC-ROC   → P(model ranks positive above negative)  (threshold-independent)
AUC-PR    → AUC-ROC's better alternative under class imbalance
RMSE/MAE  → regression error in original units
R²        → fraction of variance explained  (inflates with features — use Adjusted R²)
```

## Running Example

**Classification:** The Ch.2 classifier — is a California district high-value?  
**Regression:** The Ch.1 regressor — predict median house value from all 8 features.

Same dataset, same models as before. This chapter is pure evaluation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve,
    mean_squared_error, mean_absolute_error, r2_score,
    ConfusionMatrixDisplay,
)

# ── Load California Housing ───────────────────────────────────────────────────
data        = fetch_california_housing()
X, y_reg    = data.data, data.target       # regression target
feature_names = list(data.feature_names)

# Classification target: high-value district (above median)
median_val  = np.median(y_reg)
y_cls       = (y_reg > median_val).astype(int)

(X_train, X_test,
 y_cls_train, y_cls_test,
 y_reg_train, y_reg_test) = train_test_split(
    X, y_cls, y_reg, test_size=0.2, random_state=42)

scaler      = StandardScaler()
X_train_sc  = scaler.fit_transform(X_train)
X_test_sc   = scaler.transform(X_test)

print(f"Dataset: {X.shape[0]:,} districts, {X.shape[1]} features")
print(f"Classification target: {y_cls.sum():,} high-value / {(1-y_cls).sum():,} non-high-value "
      f"(median split = {median_val:.2f} $100k)")
print(f"Train: {len(X_train):,}  Test: {len(X_test):,}")

In [ ]:
# ── Train both models ─────────────────────────────────────────────────────────
clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train_sc, y_cls_train)

reg = LinearRegression()
reg.fit(X_train_sc, y_reg_train)

# Predicted probabilities and hard labels
y_prob = clf.predict_proba(X_test_sc)[:, 1]   # P(high-value)
y_pred = (y_prob >= 0.5).astype(int)           # default threshold

y_reg_pred = reg.predict(X_test_sc)

print("Models trained. Predicted probability range:",
      f"[{y_prob.min():.3f}, {y_prob.max():.3f}]")

## Confusion Matrix

```
                 Predicted
              Positive  Negative
Actual  Pos │   TP    │   FN   │  ← Recall = TP / (TP + FN)
        Neg │   FP    │   TN   │  ← Specificity = TN / (TN + FP)
              ↑ Precision = TP / (TP + FP)
```

In [ ]:
cm = confusion_matrix(y_cls_test, y_pred)
tn, fp, fn, tp = cm.ravel()

fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=['Non-high-value', 'High-value'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix (threshold = 0.5)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"TP={tp}  FP={fp}  FN={fn}  TN={tn}")
print(f"\nDerived metrics:")
print(f"  Accuracy  = (TP+TN) / total = ({tp}+{tn}) / {len(y_pred)} = {(tp+tn)/len(y_pred):.4f}")
print(f"  Precision = TP / (TP+FP)    = {tp} / ({tp}+{fp}) = {tp/(tp+fp):.4f}")
print(f"  Recall    = TP / (TP+FN)    = {tp} / ({tp}+{fn}) = {tp/(tp+fn):.4f}")

## All Classification Metrics

$$F_1 = \frac{2 \cdot \text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

Harmonic mean — heavily penalises if either P or R is near zero.

In [ ]:
acc     = accuracy_score(y_cls_test, y_pred)
prec    = precision_score(y_cls_test, y_pred)
rec     = recall_score(y_cls_test, y_pred)
f1      = f1_score(y_cls_test, y_pred)
auc_roc = roc_auc_score(y_cls_test, y_prob)
auc_pr  = average_precision_score(y_cls_test, y_prob)

metrics = {
    'Accuracy':        acc,
    'Precision':       prec,
    'Recall':          rec,
    'F1':              f1,
    'AUC-ROC':         auc_roc,
    'AUC-PR (Avg P)':  auc_pr,
}

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['steelblue', 'steelblue', 'steelblue', 'coral', 'teal', 'teal']
bars = ax.barh(list(metrics.keys()), list(metrics.values()),
               color=colors, edgecolor='white', height=0.6)
ax.set_xlim(0, 1.05)
for bar, val in zip(bars, metrics.values()):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=10)
ax.set_xlabel('Score'); ax.set_title('Classification Metrics (threshold = 0.5)',
                                      fontsize=12, fontweight='bold')
ax.axvline(0.5, color='grey', linestyle='--', alpha=0.5)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

for name, val in metrics.items():
    print(f"{name:20s}: {val:.4f}")

## ROC Curve

Plots TPR (Recall) vs FPR as threshold $\theta$ sweeps 1 → 0.

$$\text{AUC-ROC} = P(\hat{p}(\text{positive}) > \hat{p}(\text{negative}))$$

A random classifier lies on the diagonal (AUC = 0.5).

In [ ]:
fpr, tpr, roc_thresholds = roc_curve(y_cls_test, y_prob)
prec_curve, rec_curve, pr_thresholds = precision_recall_curve(y_cls_test, y_prob)

# Find where threshold ≈ 0.5 on the ROC curve
idx_05 = np.argmin(np.abs(roc_thresholds - 0.5))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC
axes[0].plot(fpr, tpr, color='steelblue', linewidth=2.5,
             label=f'Logistic Regression (AUC = {auc_roc:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Random classifier')
axes[0].scatter(fpr[idx_05], tpr[idx_05], s=120, color='coral', zorder=5,
                label=f'θ=0.5 (FPR={fpr[idx_05]:.2f}, TPR={tpr[idx_05]:.2f})')
axes[0].fill_between(fpr, tpr, alpha=0.08, color='steelblue')
axes[0].set_xlabel('False Positive Rate (FPR)')
axes[0].set_ylabel('True Positive Rate / Recall (TPR)')
axes[0].set_title('ROC Curve'); axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

# PR
axes[1].plot(rec_curve, prec_curve, color='coral', linewidth=2.5,
             label=f'Logistic Regression (AP = {auc_pr:.3f})')
axes[1].axhline(y_cls_test.mean(), color='k', linestyle='--', alpha=0.4,
                label=f'Random (Precision = class prior = {y_cls_test.mean():.2f})')
axes[1].fill_between(rec_curve, prec_curve, alpha=0.08, color='coral')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

plt.suptitle('ROC vs Precision-Recall Curves', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## Threshold Sweep — Finding the Optimal Operating Point

The default threshold of 0.5 is rarely optimal. Sweep $\theta$ across all values
and find where F1 peaks — or where the business constraint (e.g., Recall ≥ 0.9) is met.

In [ ]:
thresholds = np.linspace(0.01, 0.99, 200)
precs_t, recs_t, f1s_t = [], [], []

for theta in thresholds:
    y_hat = (y_prob >= theta).astype(int)
    precs_t.append(precision_score(y_cls_test, y_hat, zero_division=0))
    recs_t.append(recall_score(y_cls_test, y_hat,    zero_division=0))
    f1s_t.append(f1_score(y_cls_test, y_hat,         zero_division=0))

precs_t, recs_t, f1s_t = np.array(precs_t), np.array(recs_t), np.array(f1s_t)
best_idx = np.argmax(f1s_t)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(thresholds, precs_t, label='Precision', color='steelblue', linewidth=2)
ax.plot(thresholds, recs_t,  label='Recall',    color='coral',     linewidth=2)
ax.plot(thresholds, f1s_t,   label='F1',        color='green',     linewidth=2.5)
ax.axvline(thresholds[best_idx], color='black', linestyle='--', alpha=0.6,
           label=f'Best F1 = {f1s_t[best_idx]:.4f} at θ = {thresholds[best_idx]:.2f}')
ax.axvline(0.5, color='grey', linestyle=':', alpha=0.6, label='Default θ = 0.5')
ax.set_xlabel('Decision threshold θ');  ax.set_ylabel('Score')
ax.set_title('Precision, Recall, F1 vs Decision Threshold',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Default θ=0.5:  P={prec:.4f}  R={rec:.4f}  F1={f1:.4f}")
print(f"Best θ={thresholds[best_idx]:.2f}:  "
      f"P={precs_t[best_idx]:.4f}  R={recs_t[best_idx]:.4f}  F1={f1s_t[best_idx]:.4f}")

## Regression Metrics

| Metric | Formula | Notes |
|---|---|---|
| RMSE | $\sqrt{\frac{1}{n}\sum(\hat{y}-y)^2}$ | Same units as target; sensitive to outliers |
| MAE  | $\frac{1}{n}\sum|\hat{y}-y|$ | Robust to outliers |
| MAPE | $\frac{100}{n}\sum|\frac{\hat{y}-y}{y}|$ | Scale-free %; undefined at y=0 |
| R²   | $1 - \frac{\sum(\hat{y}-y)^2}{\sum(\bar{y}-y)^2}$ | Inflates with features — use Adjusted R² |

In [ ]:
n_test, p_feat = X_test.shape
mse    = mean_squared_error(y_reg_test, y_reg_pred)
rmse   = np.sqrt(mse)
mae    = mean_absolute_error(y_reg_test, y_reg_pred)
# MAPE — guard against y=0 (none expected here since MedHouseVal > 0)
mape   = np.mean(np.abs((y_reg_pred - y_reg_test) / y_reg_test)) * 100
r2     = r2_score(y_reg_test, y_reg_pred)
adj_r2 = 1 - (1 - r2) * (n_test - 1) / (n_test - p_feat - 1)

print(f"{'Metric':<20} {'Value':>10} {'Units'}")
print("-" * 45)
print(f"{'RMSE':<20} {rmse:>10.4f}  $100k")
print(f"{'MAE':<20} {mae:>10.4f}  $100k")
print(f"{'MAPE':<20} {mape:>10.1f}  %")
print(f"{'R²':<20} {r2:>10.4f}  (fraction of variance explained)")
print(f"{'Adjusted R²':<20} {adj_r2:>10.4f}  (R² penalised for {p_feat} features)")
print(f"\nNote: R² vs Adjusted R² difference = {r2 - adj_r2:.6f} "
      f"(small because we have many samples)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Predicted vs actual
lims = [min(y_reg_test.min(), y_reg_pred.min()) - 0.1,
        max(y_reg_test.max(), y_reg_pred.max()) + 0.1]
axes[0].scatter(y_reg_test, y_reg_pred, alpha=0.15, s=10, color='steelblue')
axes[0].plot(lims, lims, 'r--', linewidth=1.5, label='Perfect prediction')
axes[0].set_xlabel('Actual ($100k)'); axes[0].set_ylabel('Predicted ($100k)')
axes[0].set_title(f'Predicted vs Actual  (RMSE={rmse:.3f}  R²={r2:.3f})')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Residuals vs fitted
residuals = y_reg_pred - y_reg_test
axes[1].scatter(y_reg_pred, residuals, alpha=0.15, s=10, color='coral')
axes[1].axhline(0, color='black', linewidth=1.5)
axes[1].set_xlabel('Predicted ($100k)'); axes[1].set_ylabel('Residual (ŷ - y) ($100k)')
axes[1].set_title('Residuals vs Fitted — check for patterns')
axes[1].grid(alpha=0.3)

plt.suptitle('Regression Diagnostics — Linear Regression on California Housing',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Residual mean: {residuals.mean():.4f}  (should be ≈ 0 for unbiased model)")
print(f"Residual std:  {residuals.std():.4f}  (= RMSE for zero-mean residuals)")
print("\nNote: the fan shape in residuals indicates heteroscedasticity —")
print("variance grows with predicted value. Linear model is underfit at high values.")

## What Can Go Wrong — Demo: Accuracy on Imbalanced Data

Simulate an imbalanced version (5% positive class) and show that accuracy is useless.

In [ ]:
# Create an artificial 5% positive-class scenario
rng = np.random.default_rng(42)
n_imb    = 2000
n_pos    = int(n_imb * 0.05)   # 5% positive
y_imb    = np.zeros(n_imb, dtype=int)
y_imb[:n_pos] = 1
rng.shuffle(y_imb)

# "Dumb" model: always predict negative
y_always_neg = np.zeros_like(y_imb)

acc_dumb  = accuracy_score(y_imb, y_always_neg)
f1_dumb   = f1_score(y_imb, y_always_neg, zero_division=0)
rec_dumb  = recall_score(y_imb, y_always_neg, zero_division=0)

print("'Always predict negative' model on 5%-positive dataset:")
print(f"  Accuracy:  {acc_dumb:.4f}  ← looks great!")
print(f"  Recall:    {rec_dumb:.4f}  ← caught exactly 0 positives")
print(f"  F1:        {f1_dumb:.4f}  ← the truth")
print("\nLesson: always check class distribution before reporting accuracy.")
print(f"Class prior: {y_imb.mean():.2f} — a dumb model gets {1-y_imb.mean():.0%} accuracy free.")

## R² Inflation — Adding Noise Features

Plain R² **always increases** (or stays flat) when you add features — even purely random noise.
Adjusted R² penalises this and should drop when noise is added.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

rng = np.random.default_rng(42)
r2_scores, adj_r2_scores, n_noise_cols = [], [], range(0, 51, 5)

for n_noise in n_noise_cols:
    noise_train = rng.normal(0, 1, (len(X_train_sc), n_noise))
    noise_test  = rng.normal(0, 1, (len(X_test_sc),  n_noise))

    X_tr_aug = np.hstack([X_train_sc, noise_train]) if n_noise > 0 else X_train_sc
    X_te_aug = np.hstack([X_test_sc,  noise_test])  if n_noise > 0 else X_test_sc

    p_total = p_feat + n_noise
    lr      = LinearRegression().fit(X_tr_aug, y_reg_train)
    preds   = lr.predict(X_te_aug)
    r2_     = r2_score(y_reg_test, preds)
    adj_r2_ = 1 - (1 - r2_) * (n_test - 1) / (n_test - p_total - 1)
    r2_scores.append(r2_); adj_r2_scores.append(adj_r2_)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(list(n_noise_cols), r2_scores,     label='R²',           color='coral',     linewidth=2)
ax.plot(list(n_noise_cols), adj_r2_scores, label='Adjusted R²',  color='steelblue', linewidth=2, linestyle='--')
ax.set_xlabel('Number of pure noise features added')
ax.set_ylabel('Score')
ax.set_title('R² vs Adjusted R² as Noise Features Are Added',
             fontsize=12, fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"0 noise features:   R²={r2_scores[0]:.4f}  Adj R²={adj_r2_scores[0]:.4f}")
print(f"50 noise features:  R²={r2_scores[-1]:.4f}  Adj R²={adj_r2_scores[-1]:.4f}")
print("R² drifts up. Adjusted R² correctly penalises the useless features.")

## Exercises

1. **Business threshold:** The platform wants **Recall ≥ 0.90** (must not miss high-value districts). Find the lowest threshold $\theta$ that achieves this. What is the Precision at that threshold?
2. **F$\beta$ score:** Implement $F_\beta$ with $\beta = 2$ (double-weight on recall). Find the threshold that maximises $F_2$. Compare it to the F1-optimal threshold.
3. **Regression outliers:** Add a residual-vs-actual plot. Identify the 5 districts where the model is most wrong (largest absolute error). What do those districts have in common in the feature space?

In [ ]:
# Exercise 1: find the lowest theta that achieves Recall >= 0.90
# TODO: iterate over thresholds from high to low.
# For each theta: compute recall_score. Stop when recall >= 0.90.
# Print: theta, precision, recall, F1 at that point.

# Exercise 2: F_beta with beta=2
# F_beta = (1 + beta^2) * P * R / (beta^2 * P + R)
# TODO: compute F2 at each threshold. Plot F1 vs F2 curves.
# Which threshold does F2 favour? Why?

# Exercise 3: largest regression errors
# TODO: compute abs_error = |y_reg_pred - y_reg_test|
# Find top-5 error indices. Print their feature values and actual vs predicted prices.
# Hint: use X_test[top5_idx] and feature_names.